# Sprint 3 — Feature Engineering y Pipeline
## Proyecto: Productividad Asesores de Negocios

---

### Objetivo del sprint

Construir el pipeline de transformación reproducible que se usará idénticamente
en train y test. El pipeline se **fitea únicamente en train** y se aplica a test
— garantía fundamental contra data leakage en la transformación.

### Decisiones de diseño tomadas antes de este sprint

| Variable | Decisión | Justificación |
|----------|----------|---------------|
| `PASE1`, `PASE15`, `PASE30` | Excluir | Leakage contemporáneo al target |
| `DIAS_MORA`, `DIAS_MORA_RANGO` | Excluir | Base/target — leakage |
| `SEMANA_ACTUAL_RANGO` | Excluir | Consecuencia del deterioro, no causa |
| `SEXO`, `EDAD`, `ESTADO_CIVIL` | Incluir con marca FAIRNESS | Análisis de equidad en Sprint 6 |
| `TIPO_BAJA` vacíos | Imputar como 'ACTIVO' | String vacío = asesor sin baja |
| `EXP_MICROFINANZAS` vacíos | Imputar como 'SD' | Sin dato disponible |
| `NIVEL_ESTUDIOS` inconsistente | Normalizar a mayúsculas | Error de captura |

### ⚠️ Regla crítica

> El `ColumnTransformer` recibe listas de columnas **definidas estáticamente**.
> Nunca usar `df.select_dtypes()` en tiempo de ejecución — si el orden de columnas
> cambia, un selector dinámico transforma la columna equivocada silenciosamente.


## 1. Configuración y carga de datos

In [1]:
import warnings
from pathlib import Path

import pandas as pd
import numpy as np
import joblib

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, LabelEncoder
from sklearn.impute import SimpleImputer

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

INTERIM    = Path('../data/interim')
PROCESSED  = Path('../data/processed')
PROCESSED.mkdir(parents=True, exist_ok=True)

# Cargar splits de Target_B (target principal)
df_train = pd.read_parquet(INTERIM / 'target_b_train.parquet')
df_test  = pd.read_parquet(INTERIM / 'target_b_test.parquet')

print(f'Train: {df_train.shape}')
print(f'Test : {df_test.shape}')
print(f'Columnas: {list(df_train.columns)}')


Train: (1603, 37)
Test : (401, 37)
Columnas: ['CODIGO_ASESOR', 'GRUPOS', 'CLIENTES', 'PRESTAMO', 'CLIENTES_PRESTAMO', 'CARTERA_HOY', 'CARTERA_SEMANT', 'INCREMENTO_CARTERA', 'ATRASO', 'TASA_PROM', 'PROVISION_HOY', 'PROVISION_SEMANT', 'GASTO_PROVISION_SEMANAL', 'CLIENTES_NUEVOS', 'DESEMBOLSO_CLIENTES_NUEVOS', 'REGION', 'SUCURSAL', 'CODIGO_SUCURSAL', 'COORDINADOR', 'PUESTO', 'PRODUCTO', 'AREA', 'SEXO', 'ESTADO_CIVIL', 'EXP_MICROFINANZAS', 'NIVEL_ESTUDIOS', 'EDAD', 'RANGO_CICLO', 'DIA_PAGO', 'TIPO_BAJA', 'MOTIVO_BAJA', 'FECHA_ALTA', 'FECHA_BAJA', 'FECHA_NACIMIENTO', 'N_SEMANAS_OBS', 'TARGET_B', 'TDNC']


## 2. Limpieza previa al pipeline

Correcciones que no encajan en un transformer de sklearn:
strings vacíos, inconsistencias de mayúsculas y categorías mal etiquetadas.
Se aplican **antes** de fitear el pipeline, de forma idéntica a train y test.


In [2]:
def limpiar_previo(df: pd.DataFrame) -> pd.DataFrame:
    '''
    Limpieza previa al pipeline de sklearn.
    Aplica correcciones de calidad detectadas en el EDA (Sprint 1).
    
    Cambios:
      - TIPO_BAJA: string vacio -> 'ACTIVO'
      - EXP_MICROFINANZAS: string vacio -> 'SD'
      - NIVEL_ESTUDIOS: normalizar a mayusculas y unificar variantes
      - MOTIVO_BAJA: string vacio -> 'ACTIVO'
    '''
    df = df.copy()

    # TIPO_BAJA: el string vacio significa asesor activo sin baja registrada
    df['TIPO_BAJA'] = df['TIPO_BAJA'].replace('', 'ACTIVO').fillna('ACTIVO')

    # EXP_MICROFINANZAS: string vacio -> SD (sin dato)
    df['EXP_MICROFINANZAS'] = df['EXP_MICROFINANZAS'].replace('', 'SD').fillna('SD')

    # MOTIVO_BAJA: string vacio -> ACTIVO
    df['MOTIVO_BAJA'] = df['MOTIVO_BAJA'].replace('', 'ACTIVO').fillna('ACTIVO')

    # NIVEL_ESTUDIOS: normalizar a mayusculas y unificar 'Secundaria' -> 'SECUNDARIA'
    if 'NIVEL_ESTUDIOS' in df.columns:
        df['NIVEL_ESTUDIOS'] = (
            df['NIVEL_ESTUDIOS']
            .fillna('SD')
            .str.upper()
            .str.strip()
        )

    # ESTADO_CIVIL: normalizar
    if 'ESTADO_CIVIL' in df.columns:
        df['ESTADO_CIVIL'] = (
            df['ESTADO_CIVIL']
            .fillna('SD')
            .str.upper()
            .str.strip()
        )

    # SEXO: normalizar
    if 'SEXO' in df.columns:
        df['SEXO'] = df['SEXO'].fillna('SD').str.upper().str.strip()

    return df


# Aplicar limpieza a train y test
df_train = limpiar_previo(df_train)
df_test  = limpiar_previo(df_test)

# Verificar que la limpieza funcionó
print('Verificacion post-limpieza:')
for col in ['TIPO_BAJA', 'EXP_MICROFINANZAS', 'NIVEL_ESTUDIOS', 'ESTADO_CIVIL', 'SEXO']:
    if col in df_train.columns:
        vacios = (df_train[col] == '').sum()
        nulos  = df_train[col].isna().sum()
        vals   = df_train[col].nunique()
        print(f'  {col:<25} vacios={vacios} | nulos={nulos} | valores_unicos={vals}')


Verificacion post-limpieza:
  TIPO_BAJA                 vacios=0 | nulos=0 | valores_unicos=4
  EXP_MICROFINANZAS         vacios=0 | nulos=0 | valores_unicos=4
  NIVEL_ESTUDIOS            vacios=0 | nulos=0 | valores_unicos=7
  ESTADO_CIVIL              vacios=0 | nulos=0 | valores_unicos=6
  SEXO                      vacios=0 | nulos=0 | valores_unicos=3


## 3. Definición estática de columnas por tipo

Las listas de columnas se definen explícitamente aquí.
El `ColumnTransformer` las recibe como listas fijas — nunca inferidas del DataFrame.

Esto garantiza que si el DataFrame cambia de orden, el pipeline sigue
transformando las columnas correctas.


In [3]:
# ── Columnas excluidas del modelo ────────────────────────────────────────────
# Documentadas con justificación explícita
EXCLUIR = [
    'TARGET_B',           # Es el target
    'TDNC',               # Base del target — leakage
    'DIAS_MORA',          # Base del target original — leakage
    'PASE1',              # Leakage contemporáneo
    'PASE15',             # Leakage contemporáneo
    'PASE30',             # Leakage contemporáneo
    'DIAS_MORA_RANGO',    # Target_A — no usar en Target_B
    'SEMANA_ACTUAL_RANGO',# Consecuencia del deterioro, no causa
    'ASESOR',             # Identificador textual
    'NOMBRE',             # Redundante con ASESOR
    'COORDINADOR',        # Alta cardinalidad, poco predictivo
    'NUMERO_EMPLEADO',    # Redundante con CODIGO_ASESOR
    'PRIMERA_OBS',        # Metadato temporal
    'ULTIMA_OBS',         # Metadato temporal
    'FECHA_ALTA',         # Alta cardinalidad como fecha
    'FECHA_BAJA',         # Alta cardinalidad como fecha
    'FECHA_NACIMIENTO',   # Alta cardinalidad como fecha (EDAD la resume)
]

# ── Variables numéricas continuas ─────────────────────────────────────────────
# Se escalan con StandardScaler para Logistic Regression
# LightGBM y XGBoost no requieren escalado pero el pipeline lo incluye
# para que el baseline LR funcione correctamente
NUM_COLS = [
    'GRUPOS',
    'CLIENTES',
    'PRESTAMO',
    'CLIENTES_PRESTAMO',
    'CARTERA_HOY',
    'CARTERA_SEMANT',
    'INCREMENTO_CARTERA',
    'ATRASO',
    'TASA_PROM',
    'PROVISION_HOY',
    'PROVISION_SEMANT',
    'GASTO_PROVISION_SEMANAL',
    'CLIENTES_NUEVOS',
    'DESEMBOLSO_CLIENTES_NUEVOS',
    'N_SEMANAS_OBS',      # Proxy de antigüedad del asesor en el dataset
    'EDAD',               # ⚠️ FAIRNESS
]

# ── Variables categóricas de baja cardinalidad ────────────────────────────────
# OrdinalEncoder con handle_unknown='use_encoded_value' para categorías nuevas en test
CAT_COLS = [
    'REGION',
    'RANGO_CICLO',
    'DIA_PAGO',
    'PUESTO',
    'PRODUCTO',
    'AREA',
    'TIPO_BAJA',
    'MOTIVO_BAJA',
    'EXP_MICROFINANZAS',
    'NIVEL_ESTUDIOS',
    'SEXO',               # ⚠️ FAIRNESS
    'ESTADO_CIVIL',       # ⚠️ FAIRNESS
]

# Verificar que todas las columnas existen en el DataFrame
todas = NUM_COLS + CAT_COLS
faltantes = [c for c in todas if c not in df_train.columns]
if faltantes:
    print(f'ADVERTENCIA — columnas faltantes en train: {faltantes}')
else:
    print(f'Columnas numéricas  : {len(NUM_COLS)}')
    print(f'Columnas categóricas: {len(CAT_COLS)}')
    print(f'Total features      : {len(todas)}')
    print(f'Excluidas           : {len(EXCLUIR)}')


Columnas numéricas  : 16
Columnas categóricas: 12
Total features      : 28
Excluidas           : 17


## 4. Construcción del pipeline

El pipeline usa `ColumnTransformer` para aplicar transformaciones distintas
según el tipo de variable.

**Estrategia de imputación:**
- Numéricos: mediana (robusta a outliers)
- Categóricos: constante 'SD' (sin dato) — preserva la información de ausencia

**Nota sobre escalado:**
LightGBM y XGBoost son invariantes al escalado, pero el pipeline incluye
`StandardScaler` para que el baseline de Logistic Regression funcione
correctamente con las mismas features.


In [4]:
def build_pipeline() -> Pipeline:
    '''
    Construye el pipeline de preprocesamiento para Target_B.
    
    Transformaciones:
      - Numéricos: imputacion por mediana + StandardScaler
      - Categoricos: imputacion por constante 'SD' + OrdinalEncoder
    
    El pipeline se fitea UNICAMENTE en train.
    Aplicarlo a test no actualiza ningún parámetro interno.
    '''
    # Transformer para variables numéricas
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(
            strategy='median',   # Mediana: robusta a outliers detectados en Sprint 1
        )),
        ('scaler', StandardScaler()),  # Necesario para Logistic Regression baseline
    ])

    # Transformer para variables categóricas
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(
            strategy='constant',
            fill_value='SD',     # SD = sin dato — preserva ausencia como categoría
        )),
        ('encoder', OrdinalEncoder(
            handle_unknown='use_encoded_value',
            unknown_value=-1,    # Categorías nuevas en test -> -1
        )),
    ])

    # ColumnTransformer con columnas definidas estaticamente
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer,    NUM_COLS),
            ('cat', categorical_transformer, CAT_COLS),
        ],
        remainder='drop',    # Cualquier columna no listada se descarta
        verbose_feature_names_out=True,
    )

    return preprocessor


# Construir el pipeline
preprocessor = build_pipeline()
print('Pipeline construido correctamente.')
print(f'Transformers: {[t[0] for t in preprocessor.transformers]}')


Pipeline construido correctamente.
Transformers: ['num', 'cat']


## 5. Preparar X e y — separar features del target

In [5]:
# Separar features y target
TARGET_COL = 'TARGET_B'

# Filtrar columnas que existen en el DataFrame
feature_cols = [c for c in NUM_COLS + CAT_COLS if c in df_train.columns]

X_train = df_train[feature_cols].copy()
y_train = df_train[TARGET_COL].copy()

X_test  = df_test[feature_cols].copy()
y_test  = df_test[TARGET_COL].copy()

# Codificar el target como entero para sklearn
# Q1_BAJO=0, Q2_MEDIO_BAJO=1, Q3_MEDIO_ALTO=2, Q4_ALTO=3
label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train)
y_test_enc  = label_encoder.transform(y_test)

print(f'X_train: {X_train.shape}')
print(f'X_test : {X_test.shape}')
print(f'Clases del target: {list(label_encoder.classes_)}')
print(f'Encoding: {dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))}')

# Verificar que el pipeline NO se ha fiteado aun
print(f'\nPipeline fiteado: {hasattr(preprocessor, "n_features_in_")}')


X_train: (1603, 28)
X_test : (401, 28)
Clases del target: ['Q1_BAJO', 'Q2_MEDIO_BAJO', 'Q3_MEDIO_ALTO', 'Q4_ALTO']
Encoding: {'Q1_BAJO': np.int64(0), 'Q2_MEDIO_BAJO': np.int64(1), 'Q3_MEDIO_ALTO': np.int64(2), 'Q4_ALTO': np.int64(3)}

Pipeline fiteado: False


## 6. Fitear en train — transformar train y test

**Regla fundamental:** `fit_transform` solo en train.
`transform` en test — nunca `fit_transform` en test.

Usar `fit_transform` en test filtraría estadísticos del conjunto de evaluación
(mediana, media, desvio estándar) hacia el pipeline — leakage de transformación.


In [6]:
# PASO 1: Fitear el pipeline en train
# Aqui se calculan: medianas de imputacion, media y std del scaler,
# vocabulario de categorias del OrdinalEncoder
X_train_processed = preprocessor.fit_transform(X_train)

# PASO 2: Aplicar el pipeline ya fiteado a test
# NO se recalcula nada — se usan los parametros aprendidos en train
X_test_processed  = preprocessor.transform(X_test)

# Obtener nombres de features tras la transformacion
feature_names_out = preprocessor.get_feature_names_out()

print(f'X_train procesado: {X_train_processed.shape}')
print(f'X_test  procesado: {X_test_processed.shape}')
print(f'Features de salida: {len(feature_names_out)}')
print(f'\nPrimeras 10 features: {list(feature_names_out[:10])}')

# Verificar que no hay NaN tras la transformacion
nan_train = np.isnan(X_train_processed).sum()
nan_test  = np.isnan(X_test_processed).sum()
print(f'\nNaN en train procesado: {nan_train}')
print(f'NaN en test  procesado: {nan_test}')
if nan_train == 0 and nan_test == 0:
    print('Sin NaN tras transformacion')
else:
    print('ADVERTENCIA: NaN detectados — revisar imputadores')


X_train procesado: (1603, 28)
X_test  procesado: (401, 28)
Features de salida: 28

Primeras 10 features: ['num__GRUPOS', 'num__CLIENTES', 'num__PRESTAMO', 'num__CLIENTES_PRESTAMO', 'num__CARTERA_HOY', 'num__CARTERA_SEMANT', 'num__INCREMENTO_CARTERA', 'num__ATRASO', 'num__TASA_PROM', 'num__PROVISION_HOY']

NaN en train procesado: 0
NaN en test  procesado: 0
Sin NaN tras transformacion


## 7. Verificaciones de integridad del pipeline

Conjunto de pruebas que deben pasar antes de continuar al Sprint 4.
Si alguna falla, el pipeline tiene un problema que debe corregirse.


In [7]:
print('=== VERIFICACIONES DE INTEGRIDAD DEL PIPELINE ===\n')
errores = []

# Test 1: dimensiones correctas
assert X_train_processed.shape[1] == X_test_processed.shape[1], 'Dimensiones distintas'
print(f'[OK] Test 1: train y test tienen el mismo numero de features ({X_train_processed.shape[1]})')

# Test 2: sin NaN
assert np.isnan(X_train_processed).sum() == 0, 'NaN en train procesado'
assert np.isnan(X_test_processed).sum()  == 0, 'NaN en test procesado'
print('[OK] Test 2: sin valores NaN en train ni test procesados')

# Test 3: el pipeline no fue fiteado en test
# Verificar que las medianas del imputer se calcularon sobre train
mediana_train_cartera = preprocessor.named_transformers_['num']['imputer'].statistics_[
    NUM_COLS.index('CARTERA_HOY')
]
mediana_real = np.nanmedian(X_train['CARTERA_HOY'])
assert abs(mediana_train_cartera - mediana_real) < 1.0, 'Mediana del imputer no coincide con train'
print(f'[OK] Test 3: mediana de CARTERA_HOY en pipeline = {mediana_train_cartera:,.2f} (calculada en train)')

# Test 4: distribucion de y_train y y_test
clases_train = set(y_train_enc)
clases_test  = set(y_test_enc)
assert clases_train == clases_test, 'Clases distintas entre train y test'
print(f'[OK] Test 4: mismas clases en train y test: {sorted(clases_train)}')

# Test 5: features categoricas — sin valores -1 en train (no debe haber unknowns en train)
cat_start = len(NUM_COLS)
cat_block_train = X_train_processed[:, cat_start:]
n_unknowns_train = (cat_block_train == -1).sum()
if n_unknowns_train > 0:
    print(f'[WARN] Test 5: {n_unknowns_train} valores unknown (-1) en categoricas de train')
else:
    print('[OK] Test 5: sin valores unknown en categoricas de train')

print('\nTodos los tests pasaron correctamente.')


=== VERIFICACIONES DE INTEGRIDAD DEL PIPELINE ===

[OK] Test 1: train y test tienen el mismo numero de features (28)
[OK] Test 2: sin valores NaN en train ni test procesados
[OK] Test 3: mediana de CARTERA_HOY en pipeline = 53,569.65 (calculada en train)
[OK] Test 4: mismas clases en train y test: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
[OK] Test 5: sin valores unknown en categoricas de train

Todos los tests pasaron correctamente.


## 8. Guardar pipeline y datos procesados

El pipeline serializado con `joblib` se carga en los sprints de modelado
sin necesidad de re-fitearlo. Los arrays numpy procesados se guardan en
formato `.npz` para carga rápida.


In [8]:
import joblib

# Guardar el pipeline fiteado
joblib.dump(preprocessor,   PROCESSED / 'preprocessor.joblib')
joblib.dump(label_encoder,  PROCESSED / 'label_encoder.joblib')

# Guardar arrays procesados
np.savez(
    PROCESSED / 'features_processed.npz',
    X_train=X_train_processed,
    X_test=X_test_processed,
    y_train=y_train_enc,
    y_test=y_test_enc,
    feature_names=feature_names_out,
)

# Guardar lista de features para referencia en SHAP
pd.Series(feature_names_out).to_csv(PROCESSED / 'feature_names.csv', index=False)

print('Archivos guardados en data/processed/:')
for f in sorted(PROCESSED.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:<40} {size_kb:>8.1f} KB')

print('\nResumen final del pipeline:')
print(f'  Features de entrada : {len(feature_cols)}')
print(f'  Features de salida  : {X_train_processed.shape[1]}')
print(f'  Observaciones train : {X_train_processed.shape[0]:,}')
print(f'  Observaciones test  : {X_test_processed.shape[0]:,}')
print(f'  Clases del target   : {list(label_encoder.classes_)}')


Archivos guardados en data/processed/:
  .gitkeep                                      0.0 KB
  feature_names.csv                             0.5 KB
  features_processed.npz                      455.9 KB
  label_encoder.joblib                          0.5 KB
  preprocessor.joblib                           7.9 KB

Resumen final del pipeline:
  Features de entrada : 28
  Features de salida  : 28
  Observaciones train : 1,603
  Observaciones test  : 401
  Clases del target   : ['Q1_BAJO', 'Q2_MEDIO_BAJO', 'Q3_MEDIO_ALTO', 'Q4_ALTO']


## 9. Registro formal de decisiones del sprint

In [9]:
print('''
╔══════════════════════════════════════════════════════════════════╗
║         DECISIONES DE DISEÑO — SPRINT 3                        ║
║         Feature Engineering y Pipeline                         ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  VARIABLES INCLUIDAS                                             ║
║  → 16 numéricas (mediana + StandardScaler)                      ║
║  → 12 categóricas (constante SD + OrdinalEncoder)               ║
║  → SEXO, EDAD, ESTADO_CIVIL incluidas con marca FAIRNESS        ║
║                                                                  ║
║  VARIABLES EXCLUIDAS                                             ║
║  → PASE1/15/30: leakage contemporaneo                           ║
║  → SEMANA_ACTUAL_RANGO: consecuencia del deterioro              ║
║  → DIAS_MORA / DIAS_MORA_RANGO: base/target leakage             ║
║  → Fechas como fecha: alta cardinalidad sin valor predictivo    ║
║  → Identificadores textuales: ASESOR, NOMBRE, COORDINADOR       ║
║                                                                  ║
║  IMPUTACION                                                      ║
║  → Numéricos: mediana (robusta a outliers)                      ║
║  → Categoricos: constante SD (preserva ausencia)               ║
║                                                                  ║
║  ENCODING                                                        ║
║  → OrdinalEncoder con unknown_value=-1                          ║
║  → Categorias nuevas en test -> -1 (no rompe el pipeline)      ║
║                                                                  ║
║  ESCALADO                                                        ║
║  → StandardScaler incluido para Logistic Regression baseline    ║
║  → LightGBM/XGBoost son invariantes al escalado                 ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
''')



╔══════════════════════════════════════════════════════════════════╗
║         DECISIONES DE DISEÑO — SPRINT 3                        ║
║         Feature Engineering y Pipeline                         ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  VARIABLES INCLUIDAS                                             ║
║  → 16 numéricas (mediana + StandardScaler)                      ║
║  → 12 categóricas (constante SD + OrdinalEncoder)               ║
║  → SEXO, EDAD, ESTADO_CIVIL incluidas con marca FAIRNESS        ║
║                                                                  ║
║  VARIABLES EXCLUIDAS                                             ║
║  → PASE1/15/30: leakage contemporaneo                           ║
║  → SEMANA_ACTUAL_RANGO: consecuencia del deterioro              ║
║  → DIAS_MORA / DIAS_MORA_RANGO: base/target leakage             ║
║  → Fechas como fecha: alta cardinalidad s